# Comparación en GPU (Colab, T4 gratis)

Corre **el mismo código del repo** en una GPU: `pipeline.py bench` transcribe con la rejilla de `config.yaml` (modelos × dispositivo × precisión) y guarda en cada corrida el dispositivo **leído de CTranslate2 en tiempo de ejecución** y el **pico de VRAM medido con NVML**.

Entorno → Cambiar tipo de entorno → **T4 GPU** antes de empezar.

In [1]:
!nvidia-smi
!git clone https://github.com/luyov-belli/HW05_DS.git
%cd HW05_DS
!pip install -q -r requirements.txt

Mon Nov  2 02:15:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# CTranslate2 4.8 está compilado para CUDA 12. Si Colab trae una CUDA más nueva, falta
# libcublas.so.12 ("Library libcublas.so.12 is not found"): se instalan cuBLAS 12 y cuDNN 9
# como paquetes de pip y se le dice al sistema dónde están. Los `!python` heredan esta variable.
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
import glob, os, site
libs = {os.path.dirname(p) for d in site.getsitepackages()
        for name in ("libcublas.so.12", "libcudnn.so.9")
        for p in glob.glob(f"{d}/nvidia/**/{name}", recursive=True)}
os.environ["LD_LIBRARY_PATH"] = ":".join(sorted(libs) + [os.environ.get("LD_LIBRARY_PATH", "")])
print(libs)
!python -c "import ctypes; ctypes.CDLL('libcublas.so.12'); ctypes.CDLL('libcudnn.so.9'); print('CUDA 12 OK')"

{'/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib', '/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib'}
CUDA 12 OK


## Audio
Sube tus grabaciones a `MyDrive/HW05_audio/` con los nombres del guion (`E01.m4a`, …, `L05.m4a`). Si aún no grabas, la segunda línea genera el corpus sintético.

In [3]:
CORPUS = "real"   # o "tts-es-PE"
if CORPUS == "real":
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p data/audio/real && cp /content/drive/MyDrive/HW05_audio/* data/audio/real/
else:
    !python pipeline.py tts --locales es-PE

Mounted at /content/drive


In [4]:
# Rejilla GPU/CPU/precisión + glosario con el modelo grande
!python pipeline.py bench --corpus {CORPUS}
# Escalera de las llamadas con el modelo grande en GPU
!python pipeline.py ladder --corpus {CORPUS}
!python pipeline.py transcribe --corpus {CORPUS} --model large-v3 --device cuda --task pedido --level L0 L1 L2 L3

  E01: 4.97 s de ASR para 137.83 s de audio
  E02: 4.08 s de ASR para 141.33 s de audio
  E03: 3.76 s de ASR para 102.42 s de audio
  E04: 3.36 s de ASR para 104.21 s de audio
real__entrevista__L0__small__cuda__float16__glosario: device=cuda vram=887 MB 2.0 s/min
  E01: 4.66 s de ASR para 137.83 s de audio
  E02: 4.17 s de ASR para 141.33 s de audio
  E03: 4.08 s de ASR para 102.42 s de audio
  E04: 3.33 s de ASR para 104.21 s de audio
real__entrevista__L0__small__cuda__int8__glosario: device=cuda vram=522 MB 2.01 s/min
  E01: 13.95 s de ASR para 137.83 s de audio
  E02: 13.85 s de ASR para 141.33 s de audio
  E03: 12.0 s de ASR para 102.42 s de audio
  E04: 10.01 s de ASR para 104.21 s de audio
real__entrevista__L0__large-v3__cuda__float16__glosario: device=cuda vram=4106 MB 6.15 s/min
  E01: 14.21 s de ASR para 137.83 s de audio
  E02: 20.62 s de ASR para 141.33 s de audio
  E03: 11.95 s de ASR para 102.42 s de audio
  E04: 9.92 s de ASR para 104.21 s de audio
real__entrevista__L0__l

In [5]:
# Evidencia: dispositivo leído en runtime y VRAM medida de cada corrida
import json, glob
for f in sorted(glob.glob("results/transcripts/*.json")):
    m = json.load(open(f))["meta"]
    print(f"{m['model']:>9} {m['device_runtime']:>4} {m['compute_type_runtime']:>13} glos={m['glossary']!s:5} VRAM={m['vram_peak_mb']} MB  {m['seconds_per_audio_minute']} s/min  {m['gpu']}")

     base  cpu  int8_float32 glos=True  VRAM=None MB  6.78 s/min  None
 large-v3 cuda       float16 glos=True  VRAM=4106 MB  6.15 s/min  Tesla T4
 large-v3 cuda       float16 glos=False VRAM=4874 MB  6.66 s/min  Tesla T4
 large-v3 cuda  int8_float16 glos=True  VRAM=3370 MB  7.0 s/min  Tesla T4
    small  cpu       float32 glos=True  VRAM=None MB  28.89 s/min  None
    small  cpu  int8_float32 glos=True  VRAM=0 MB  33.99 s/min  Tesla T4
    small  cpu  int8_float32 glos=False VRAM=None MB  14.92 s/min  None
    small cuda       float16 glos=True  VRAM=887 MB  2.0 s/min  Tesla T4
    small cuda  int8_float16 glos=True  VRAM=522 MB  2.01 s/min  Tesla T4
 large-v3 cuda       float16 glos=True  VRAM=4151 MB  6.21 s/min  Tesla T4
    small  cpu  int8_float32 glos=True  VRAM=None MB  27.35 s/min  None
    small  cpu  int8_float32 glos=False VRAM=None MB  15.19 s/min  None
 large-v3 cuda       float16 glos=True  VRAM=4042 MB  6.27 s/min  Tesla T4
    small  cpu  int8_float32 glos=True  VRAM=No

In [6]:
# Descarga las transcripciones cacheadas y cópialas a results/transcripts/ de tu repo local
!zip -qr transcripts_gpu.zip results/transcripts results/ladder_*.json
from google.colab import files
files.download("transcripts_gpu.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>